In [1]:
from script.OCR import ocr
import cv2
from ultralytics import YOLO
import glob
import os
from paddleocr import PaddleOCR
import numpy as np

**YOLO Model**

In [ ]:
model = YOLO("models/best_obb.pt")

test_images = glob.glob("test_images/*.jpg") + glob.glob("test_images/*.png")
print(f"Found {len(test_images)} test images")

results = model.predict(
    source=test_images,
    save=True,
    project=r"J:\Python\MVP\outputs",
    name="predict_obb_best",
    exist_ok=True,
    conf=0.5,
    iou=0.4,
)

**Cropping**

In [ ]:
output_dir = r"J:\Python\MVP\outputs\crops_obb"
os.makedirs(output_dir, exist_ok=True)


def order_points(pts):
    """
    Order points as:
    top-left, top-right, bottom-right, bottom-left
    """
    pts = np.array(pts, dtype=np.float32)

    rect = np.zeros((4, 2), dtype=np.float32)

    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]      # Top-left
    rect[2] = pts[np.argmax(s)]      # Bottom-right

    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]   # Top-right
    rect[3] = pts[np.argmax(diff)]   # Bottom-left

    return rect



for img_path, result in zip(test_images, results):
    img = cv2.imread(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]

    obb_corners = result.obb.xyxyxyxy.cpu().numpy()

    for i, corners in enumerate(obb_corners):

        # -------------------------------
        # Reorder the corners correctly
        # -------------------------------
        corners = order_points(corners)


        # -------------------------------
        # Compute width
        # -------------------------------
        widthA = np.linalg.norm(corners[2] - corners[3])
        widthB = np.linalg.norm(corners[1] - corners[0])
        width = int(max(widthA, widthB))

        # -------------------------------
        # Compute height
        # -------------------------------
        heightA = np.linalg.norm(corners[1] - corners[2])
        heightB = np.linalg.norm(corners[0] - corners[3])
        height = int(max(heightA, heightB))

        if width < 2 or height < 2:
            continue

        dst = np.array([
            [0, 0],
            [width - 1, 0],
            [width - 1, height - 1],
            [0, height - 1]
        ], dtype=np.float32)


        # -------------------------------
        # Perspective transform
        # -------------------------------
        M = cv2.getPerspectiveTransform(corners, dst)

        crop = cv2.warpPerspective(img, M, (width, height), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

        # -------------------------------
        # Rotate if vertical
        # -------------------------------
        # h, w = crop.shape[:2]
        # if h > w:
        #     crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)

        

        crop_filename = os.path.join(output_dir, f"{img_name}_spine{i}.jpg")
        cv2.imwrite(crop_filename, crop)

In [2]:
# Initialize PaddleOCR
paddle_ocr = PaddleOCR(
    enable_mkldnn=False,
    use_doc_orientation_classify=True,
    use_doc_unwarping=False,
    use_textline_orientation=True
)

j:\Python\MVP\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\DELL\.paddlex\official_models\PP-OCRv6_medium_det`.
Creating model: ('P

In [31]:
paths = glob.glob("outputs/crops_obb/*.jpg")
os.makedirs("outputs/pre_obb", exist_ok=True)

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
target_height = 256
max_width = 1000
max_scale = 3

for path in paths:
    img = cv2.imread(path)

    # Convert to grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE
    gray = clahe.apply(gray)

    # Resizing
    h, w = gray.shape[:2]

    scale = target_height / h
    if scale > max_scale:
        scale = max_scale

    new_w = int(w * scale)
    new_h = int(h * scale)

    if new_w > max_width:
        scale = max_width / w
        new_w = max_width
        new_h = int(h * scale)


    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC

    resized = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=interpolation)

    filename = os.path.basename(path)
    print(f"{filename}: {resized.shape}")
    cv2.imwrite(os.path.join("outputs/pre_obb", filename), resized)

img1_spine0.jpg: (256, 40)
img1_spine1.jpg: (256, 56)
img1_spine2.jpg: (256, 44)
img1_spine3.jpg: (256, 81)
img1_spine4.jpg: (97, 1000)
img1_spine5.jpg: (135, 1000)
img2_spine0.jpg: (256, 52)
img2_spine1.jpg: (256, 37)
img2_spine10.jpg: (77, 1000)
img2_spine2.jpg: (149, 1000)
img2_spine3.jpg: (256, 23)
img2_spine4.jpg: (256, 46)
img2_spine5.jpg: (80, 1000)
img2_spine6.jpg: (256, 21)
img2_spine7.jpg: (256, 15)
img2_spine8.jpg: (256, 20)
img2_spine9.jpg: (256, 23)
IMG_20260722_090255_spine0.jpg: (256, 42)
IMG_20260722_090255_spine1.jpg: (134, 1000)
IMG_20260722_090255_spine2.jpg: (109, 1000)
IMG_20260722_090255_spine3.jpg: (256, 85)
IMG_20260722_090255_spine4.jpg: (256, 45)
IMG_20260722_090255_spine5.jpg: (256, 56)
IMG_20260722_090305_spine0.jpg: (256, 53)
IMG_20260722_090305_spine1.jpg: (256, 36)
IMG_20260722_090305_spine10.jpg: (73, 1000)
IMG_20260722_090305_spine11.jpg: (256, 21)
IMG_20260722_090305_spine12.jpg: (256, 220)
IMG_20260722_090305_spine2.jpg: (147, 1000)
IMG_20260722_09030

In [37]:
path = 'outputs/pre_obb/img2_spine5.jpg'

img = cv2.imread(path)
ocr_results = paddle_ocr.predict(img)

img_name = os.path.splitext(os.path.basename(path))[0]

print(f'Image: {img_name}')

for res in ocr_results:
    texts = res['rec_texts']
    scores = res['rec_scores']

    for text, score in zip(texts, scores):
        print(f'Text: {text:30} Confidence: {score:.4f}')

Image: img2_spine5
Text: MARK                           Confidence: 1.0000
Text: THE SUBTLE ART OF              Confidence: 0.9999
Text: 子                              Confidence: 0.2651
Text: MANSON                         Confidence: 1.0000
Text: NOT GIVING A FPCK              Confidence: 0.9620


In [38]:
paths = glob.glob("outputs/pre_obb/*.jpg")

for path in paths:
    img = cv2.imread(path)
    ocr_results = paddle_ocr.predict(img)

    img_name = os.path.splitext(os.path.basename(path))[0]

    print(f'Image: {img_name}')

    for res in ocr_results:
        texts = res['rec_texts']
        scores = res['rec_scores']

        for text, score in zip(texts, scores):
            print(f'Text: {text:30} Confidence: {score:.4f}')

Image: img1_spine0
Image: img1_spine1
Image: img1_spine2
Text: 2D                             Confidence: 0.3435
Text: 22010                          Confidence: 0.3851
Text: rlmmail                        Confidence: 0.4465
Image: img1_spine3
Text: 0X2O                           Confidence: 0.3462
Image: img1_spine4
Text:                                Confidence: 0.0000
Image: img1_spine5
Text:                                Confidence: 0.0000
Text: QASIM ALI SHAH                 Confidence: 0.9999
Text: FOUNDATION                     Confidence: 1.0000
Image: img2_spine0
Text: Yuval                          Confidence: 0.8565
Text: Noah                           Confidence: 0.9878
Text: Hararl                         Confidence: 0.9293
Text: Homo Deus                      Confidence: 0.8999
Text: Jaugy                          Confidence: 0.7774
Text: yo i                           Confidence: 0.7565
Image: img2_spine1
Text: HARRYPOTTER                    Confidence: 0.9971
Text: HE